# Where Am I? — Try it in PyTorch

This is an **optional** hands-on companion to [Chapter 8: Where Am I?](https://learnai.robennals.org/positions). I'll start by re-creating the toy attention from the previous notebook, see that it can't tell word order apart, then add positions two ways: **ALiBi** (linear distance penalty) and **RoPE** (rotation by position). I'll explore multi-speed RoPE for different attention-distance shapes, and finish with **causal masking** to enforce "only look backward".

*This notebook reuses the chapter 7 attention toy. If you haven't gone through [Paying Attention](https://learnai.robennals.org/attention) yet, I recommend that first.*

*New to PyTorch? See the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a beginner-friendly introduction.*

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math

## Recap: the toy attention from chapter 7

I'll rebuild the same 4-token toy: cat, dog, blah, it. Each token has a 1D key, query, and a value vector. Then I'll feed the same tokens to attention in two different orders and check whether the output changes — it shouldn't, because the toy attention is position-blind.

In [ ]:
TOKEN_KEY = {'cat': 3.0, 'dog': 3.0, 'blah': 0.0, 'it': 0.0}
TOKEN_QUERY = {'cat': 0.0, 'dog': 0.0, 'blah': 0.0, 'it': 3.0}
TOKEN_VALUE = {
    'cat':  torch.tensor([1.0, 0.0]),
    'dog':  torch.tensor([0.0, 1.0]),
    'blah': torch.tensor([0.0, 0.0]),
    'it':   torch.tensor([0.0, 0.0]),
}

def softmax(scores):
    scores = torch.tensor(scores, dtype=torch.float32)
    m = scores.max()
    e = torch.exp(scores - m)
    return (e / e.sum()).tolist()

def attend(sentence, focus_index):
    """Compute attention output for sentence[focus_index]. Returns blended value vector."""
    q = TOKEN_QUERY[sentence[focus_index]]
    scores = [q * TOKEN_KEY[tok] for tok in sentence]
    weights = softmax(scores)
    blended = torch.zeros(2)
    for tok, w in zip(sentence, weights):
        blended += w * TOKEN_VALUE[tok]
    return weights, blended

# Same tokens in two different orders, focus on 'it' both times
for sent in [
    ['cat', 'blah', 'dog', 'it'],
    ['dog', 'blah', 'cat', 'it'],
]:
    weights, blended = attend(sent, len(sent) - 1)
    print(f'\nsentence: {sent}')
    print(f'  attention weights: {[round(w, 3) for w in weights]}')
    print(f'  blended value:     [cat-ness {blended[0]:.3f}, dog-ness {blended[1]:.3f}]')

print('\nIdentical blended values: position-blindness confirmed.')

## ALiBi: subtract a linear distance penalty

The simplest way to give attention a sense of position: subtract a penalty proportional to distance from each attention score. With slope `m`, the new score is `(query · key) − m × distance`. At slope 0 the model is position-blind. At higher slopes, the closer noun gets more attention than the farther one — even when both have the same key.

In [ ]:
def attend_alibi(sentence, focus_index, slope):
    """Attention with linear distance penalty (ALiBi)."""
    q = TOKEN_QUERY[sentence[focus_index]]
    scores = []
    for i, tok in enumerate(sentence):
        dot = q * TOKEN_KEY[tok]
        distance = abs(focus_index - i)
        scores.append(dot - slope * distance)
    weights = softmax(scores)
    blended = torch.zeros(2)
    for tok, w in zip(sentence, weights):
        blended += w * TOKEN_VALUE[tok]
    return weights, blended

sentence = ['cat', 'blah', 'blah', 'dog', 'it']  # cat is far, dog is close to 'it'
for slope in [0.0, 0.5, 1.5]:
    weights, blended = attend_alibi(sentence, len(sentence) - 1, slope)
    print(f'\nslope={slope}')
    for tok, w in zip(sentence, weights):
        bar = '#' * int(w * 40)
        print(f'  {tok:6s} {w:.3f}  {bar}')
    print(f'  blended: cat-ness {blended[0]:.3f}, dog-ness {blended[1]:.3f}')

print("\nAt slope 0, cat and dog tie. At higher slope, dog (closer) wins.")

## Rotation Encodes Distance

There's a completely different way to encode position. Start with this fact: **rotating two vectors by the same angle leaves their dot product unchanged**. The dot product only cares about the *gap* between two vectors' directions, not where they're pointing in absolute terms.

In [ ]:
def rotate_2d(v, angle_rad):
    c = math.cos(angle_rad)
    s = math.sin(angle_rad)
    return torch.tensor([c * v[0] - s * v[1], s * v[0] + c * v[1]])

v1 = torch.tensor([1.0, 0.5])
v2 = torch.tensor([0.3, 0.8])
print(f"original v1={v1.tolist()}, v2={v2.tolist()}")
print(f"  dot(v1, v2) = {torch.dot(v1, v2).item():.4f}\n")

for angle_deg in [30, 90, 137]:
    a = math.radians(angle_deg)
    r1 = rotate_2d(v1, a)
    r2 = rotate_2d(v2, a)
    print(f"both rotated {angle_deg}deg:")
    print(f"  v1 -> {[round(x.item(), 3) for x in r1]}")
    print(f"  v2 -> {[round(x.item(), 3) for x in r2]}")
    print(f"  dot(r1, r2) = {torch.dot(r1, r2).item():.4f}  <-- unchanged")
    print()

# Now rotate them by DIFFERENT angles
print('Rotating by DIFFERENT angles changes the dot product:')
for delta_deg in [0, 30, 90]:
    r1 = rotate_2d(v1, 0.0)
    r2 = rotate_2d(v2, math.radians(delta_deg))
    print(f'  gap={delta_deg}deg: dot = {torch.dot(r1, r2).item():.4f}')

## Applying Rotation to a Dimension

Now connect rotation to attention: take one dimension of each token's key — the "noun-ness" I had before — and split it into two coordinates (`noun-x`, `noun-y`). Then rotate by `position × speed`. Tokens at similar positions stay similar; tokens far apart get rotated to different angles. The dot product naturally favors nearby tokens.

In [ ]:
# Each token gets a 2D 'noun pair' key. Same noun-ness as before but represented as (k, 0).
TOKEN_KEY_2D = {tok: torch.tensor([TOKEN_KEY[tok], 0.0]) for tok in TOKEN_KEY}
TOKEN_QUERY_2D = {tok: torch.tensor([TOKEN_QUERY[tok], 0.0]) for tok in TOKEN_QUERY}

def attend_rope_one_pair(sentence, focus_index, speed_deg):
    """Single-pair RoPE: rotate each token's 2D key by position * speed."""
    speed_rad = math.radians(speed_deg)
    q_focus = rotate_2d(TOKEN_QUERY_2D[sentence[focus_index]], focus_index * speed_rad)
    scores = []
    for i, tok in enumerate(sentence):
        k_rot = rotate_2d(TOKEN_KEY_2D[tok], i * speed_rad)
        scores.append(torch.dot(q_focus, k_rot).item())
    weights = softmax(scores)
    return weights

sentence = ['cat', 'blah', 'blah', 'dog', 'it']
for speed in [0, 15, 60]:
    weights = attend_rope_one_pair(sentence, len(sentence) - 1, speed)
    print(f'\nspeed={speed}deg/position')
    for tok, w in zip(sentence, weights):
        bar = '#' * int(w * 40)
        print(f'  {tok:6s} {w:.3f}  {bar}')

print('\nAt speed 0: position-blind (cat and dog tie).')
print('Higher speeds: dog (closer to it) wins, just like ALiBi — but achieved by rotation.')

## Multiple Rotation Speeds

Real RoPE uses **many** dimension pairs, each rotating at its own speed. Combining different speeds in different proportions lets the model express many distance-falloff *shapes* — sharp spikes, broad spreads, two-stage dropoffs, anything in between.

Below I implement multi-pair RoPE with 8 dimension pairs at exponentially-decreasing speeds, then plot the attention-distance curves for the chapter's four configurations.

In [ ]:
N_PAIRS = 8
MAX_DISTANCE = 32

# Speed of pair i: 30 / 2^i degrees per position (exponential decay)
speeds_deg = [30.0 / (2 ** i) for i in range(N_PAIRS)]
speeds_rad = [math.radians(s) for s in speeds_deg]
print(f'Speeds (deg/position): {[round(s, 3) for s in speeds_deg]}')

def attention_curve(weights_per_pair):
    """Compute attention weight as a function of distance.
    Each pair's contribution to the dot product is `weight * cos(distance * speed)`,
    assuming the query and key for that pair are aligned (so the only thing differing is rotation)."""
    distances = list(range(MAX_DISTANCE))
    raw_scores = []
    for d in distances:
        s = sum(w * math.cos(d * sp) for w, sp in zip(weights_per_pair, speeds_rad))
        raw_scores.append(s)
    # Softmax across distances to get attention weights
    return distances, softmax(raw_scores)

# Chapter scenarios from RoPEMultiSpeedWidget
configs = [
    ('Single fast pair only',  [1.0, 0, 0, 0, 0, 0, 0, 0]),
    ('Single slow pair only',  [0, 0, 0, 0, 0, 0, 0, 1.0]),
    ('Sharper medium combo',   [0.4, 0.4, 0.2, 0, 0, 0, 0, 0]),
    ('Two-stage dropoff',      [0.6, 0, 0, 0, 0, 0, 0, 0.4]),
]

plt.figure(figsize=(10, 6))
for label, weights in configs:
    distances, attn = attention_curve(weights)
    plt.plot(distances, attn, marker='o', label=label, linewidth=1.5)
plt.title('Multi-speed RoPE: attention weight as a function of distance')
plt.xlabel('distance (positions)')
plt.ylabel('attention weight')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Causal Masking: Only Look Backward

RoPE encodes distance, not direction — the dot product between two rotated vectors is the same whether token A is 3 positions *before* or 3 positions *after* token B. For generative language models, direction comes from **causal masking**: when the model generates token *t*, it can only attend to tokens *0..t* (the future doesn't exist yet). Future positions are masked out by setting their attention scores to `-inf` before softmax — which makes them contribute zero attention weight.

In [ ]:
# Build a small attention scores matrix (5 tokens, all attending to all)
torch.manual_seed(0)
n = 5
raw_scores = torch.randn(n, n) * 2.0
print('Raw scores (random, no causal mask):')
print(raw_scores.round(decimals=2))
print()

# Causal mask: positions to the right of the diagonal get -inf
causal_mask = torch.triu(torch.ones(n, n), diagonal=1).bool()
masked = raw_scores.masked_fill(causal_mask, float('-inf'))
print('After causal masking (-inf in upper-right triangle):')
print(masked.round(decimals=2))
print()

weights = F.softmax(masked, dim=-1)
print('Attention weights (rows sum to 1; upper-right triangle is exactly 0):')
print(weights.round(decimals=3))
print()

# Spot-check: row 2 only attends to columns 0, 1, 2
print(f'Row 2 weights:  {[round(w, 3) for w in weights[2].tolist()]}')
print(f'Row 2 sum to 3 (positions 0..2): {weights[2, :3].sum().item():.4f}')
print(f'Row 2 sum past 3 (positions 3..): {weights[2, 3:].sum().item():.4f}  (should be 0.0)')

---

*This notebook accompanies [Chapter 8: Where Am I?](https://learnai.robennals.org/positions). I've now seen the three pieces a transformer needs for attention: what each word *means* (embeddings), which other words *matter* (Q/K/V scoring), and where each word *is* (RoPE + causal masking). [Chapter 9: One Architecture to Rule Them All](https://learnai.robennals.org/transformers) wires them together.*

*New to PyTorch? See the [PyTorch appendix](https://learnai.robennals.org/appendix-pytorch) for a beginner-friendly introduction.*